In [1]:
import pandas as pd
import numpy as np
import sys
import os
sys.path.append("/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/")
from SynOmics.synthesizer.GaussianCopulasynthesizer import GaussianCopulasynthesizer
from SynOmics.processing.metadata import MetaData



fullDF = pd.read_csv("OriginalData/integrated_data_melanoma.csv", index_col = 0)

responderCrit = fullDF['BR'].isin(['CR','PR'])
progressorCrit = fullDF['BR'].isin(['PD'])

responders = fullDF[responderCrit].index
progressors = fullDF[progressorCrit].index
nonresponders = fullDF[~responderCrit].index
nonprogressors = fullDF[~progressorCrit].index

fullDF["PD vs Responder"] = "SD/MR"
fullDF.loc[responders,"PD vs Responder"] = 'Responder'
fullDF.loc[progressors,"PD vs Responder"] = 'Progressor'


In [2]:
ordinal_cat_columns = ["BR","Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext","PD vs Responder"]
metadata = MetaData.get_metadata(data = fullDF, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ordinal_cat_columns)

os.makedirs("GaussianCopulaTest", exist_ok = True)
seed = 42
output_path = f"GaussianCopulaTest/gaussiancopula{seed}_RespondersOrdinal"
synth = GaussianCopulasynthesizer(output_path=output_path, metadata=metadata)
synthetic_data = synth.generate(
data=fullDF,
data_ids=fullDF.index.tolist(),
enforce_rounding=True,
enforce_min_max=False,
masking=False,
n_samples=fullDF.shape[0],
seed=seed,
fit_params={"n_jobs": 12, "chunk_size": 1000},
output_filename=f"gaussiancopula_{seed}_RespondersOrdinal.csv",
save_index=False,
)

2025-11-26 19:42:17 - DEBUG - Standalone logger initialized successfully.
2025-11-26 19:42:17 - INFO - ========== Synthesizer Initialized ==========
2025-11-26 19:42:17 - INFO - Class: GaussianCopulasynthesizer
2025-11-26 19:42:17 - INFO - Output path: GaussianCopulaTest/gaussiancopula42_RespondersOrdinal
2025-11-26 19:42:17 - INFO - Log file: GaussianCopulaTest/gaussiancopula42_RespondersOrdinal/GaussianCopulasynthesizer_139645333117968.log
2025-11-26 19:42:17 - INFO - Metadata provided with 18815 columns.
2025-11-26 19:42:17 - INFO - ============================================
--- System & Process Info ---
Current Date and Time (UTC): 2025-11-26 18:42:18
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 48
Logical Processors: 48
Process RAM before execution: 287.32 MB

--- GPU Info ---
GPU monitoring failed. Ensure 'nvidia-ml-py' is installed and NVIDIA drivers are accessible.
Error: module 'nvidia_smi' has no attribute 'nvidia_smi_lib'

--- Function Execution ---
2025

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  19 | elapsed:  5.7min remaining: 48.1min
[Parallel(n_jobs=12)]: Done   4 out of  19 | elapsed:  5.8min remaining: 21.6min
[Parallel(n_jobs=12)]: Done   6 out of  19 | elapsed:  5.9min remaining: 12.8min
[Parallel(n_jobs=12)]: Done   8 out of  19 | elapsed:  6.0min remaining:  8.2min
[Parallel(n_jobs=12)]: Done  10 out of  19 | elapsed:  6.0min remaining:  5.4min
[Parallel(n_jobs=12)]: Done  12 out of  19 | elapsed:  6.1min remaining:  3.6min
[Parallel(n_jobs=12)]: Done  14 out of  19 | elapsed: 10.1min remaining:  3.6min
[Parallel(n_jobs=12)]: Done  16 out of  19 | elapsed: 10.3min remaining:  1.9min
[Parallel(n_jobs=12)]: Done  19 out of  19 | elapsed: 10.3min finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  19 | elapsed:    2.3s remaining:   19.8s
[Parallel(n_jobs=12)]: Done   4 out of  19 | e

2025-11-26 20:06:25 - INFO - Saved Gaussian Copula model to GaussianCopulaTest/gaussiancopula42_RespondersOrdinal/GaussianCopula_model.pkl
2025-11-26 20:06:25 - INFO - Model fit complete.
2025-11-26 20:24:15 - INFO - Generated 121 synthetic samples with Gaussian Copula
2025-11-26 20:24:15 - INFO - Sampled 121 rows.
2025-11-26 20:24:17 - INFO - Anonymizing IDs in synthetic data
2025-11-26 20:24:18 - INFO - Applying rounding to synthetic data
2025-11-26 20:24:31 - INFO - Postprocessed synthetic data. Final shape: (121, 18816)
2025-11-26 20:24:44 - INFO - Saved synthetic data to GaussianCopulaTest/gaussiancopula42_RespondersOrdinal/gaussiancopula_42_RespondersOrdinal.csv
2025-11-26 20:24:44 - INFO - Pipeline complete.

--- Resource Usage Summary ---
Execution time: 2546.753495 seconds
Process RAM after execution: 3303.02 MB
Process RAM used by function: 3015.70 MB
Average per-core CPU during execution: [92.5, 87.4, 92.3, 87.1, 90.8, 87.3, 94.0, 87.1, 96.6, 86.9, 88.0, 87.0, 87.9, 87.0, 88

In [3]:
synthetic_data["BR"].value_counts()

BR
PD    47
PR    33
SD    15
CR    14
MR    12
Name: count, dtype: int64

In [4]:
fullDF["BR"].value_counts()

BR
PD    56
PR    31
CR    16
SD    16
MR     2
Name: count, dtype: int64

In [5]:
fullDF["PD vs Responder"].value_counts()

PD vs Responder
Progressor    56
Responder     47
SD/MR         18
Name: count, dtype: int64

In [6]:
synthetic_data["PD vs Responder"].value_counts()

PD vs Responder
Progressor    56
Responder     34
SD/MR         31
Name: count, dtype: int64

In [7]:
ordinal_cat_columns = ["Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext"]
fullDF = fullDF.drop(columns = ["BR"])
metadata = MetaData.get_metadata(data = fullDF, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ordinal_cat_columns)

os.makedirs("GaussianCopulaTest", exist_ok = True)
seed = 42
output_path = f"GaussianCopulaTest/gaussiancopula{seed}_noBR"
synth = GaussianCopulasynthesizer(output_path=output_path, metadata=metadata)
synthetic_data = synth.generate(
data=fullDF,
data_ids=fullDF.index.tolist(),
enforce_rounding=True,
enforce_min_max=False,
masking=False,
n_samples=fullDF.shape[0],
seed=seed,
fit_params={"n_jobs": -1, "chunk_size": 1000},
output_filename=f"gaussiancopula_{seed}_noBR.csv",
save_index=False,
)

2025-11-26 22:16:19 - DEBUG - Standalone logger initialized successfully.
2025-11-26 22:16:19 - INFO - ========== Synthesizer Initialized ==========
2025-11-26 22:16:19 - INFO - Class: GaussianCopulasynthesizer
2025-11-26 22:16:19 - INFO - Output path: GaussianCopulaTest/gaussiancopula42_noBR
2025-11-26 22:16:19 - INFO - Log file: GaussianCopulaTest/gaussiancopula42_noBR/GaussianCopulasynthesizer_139647371394112.log
2025-11-26 22:16:19 - INFO - Metadata provided with 18814 columns.
2025-11-26 22:16:19 - INFO - ============================================
--- System & Process Info ---
Current Date and Time (UTC): 2025-11-26 21:16:19
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 48
Logical Processors: 48
Process RAM before execution: 552.52 MB

--- GPU Info ---
GPU monitoring failed. Ensure 'nvidia-ml-py' is installed and NVIDIA drivers are accessible.
Error: module 'nvidia_smi' has no attribute 'nvidia_smi_lib'

--- Function Execution ---
2025-11-26 22:16:19 - INFO - P

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  19 | elapsed:  7.4min remaining: 63.1min
[Parallel(n_jobs=-1)]: Done   4 out of  19 | elapsed:  7.6min remaining: 28.6min
[Parallel(n_jobs=-1)]: Done   6 out of  19 | elapsed:  7.7min remaining: 16.7min
[Parallel(n_jobs=-1)]: Done   8 out of  19 | elapsed:  7.8min remaining: 10.7min
[Parallel(n_jobs=-1)]: Done  10 out of  19 | elapsed:  7.8min remaining:  7.0min
[Parallel(n_jobs=-1)]: Done  12 out of  19 | elapsed:  7.8min remaining:  4.6min
[Parallel(n_jobs=-1)]: Done  14 out of  19 | elapsed:  7.8min remaining:  2.8min
[Parallel(n_jobs=-1)]: Done  16 out of  19 | elapsed:  7.9min remaining:  1.5min
[Parallel(n_jobs=-1)]: Done  19 out of  19 | elapsed:  8.0min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  19 | elapsed:    8.7s remaining:  1.2min
[Parallel(n_jobs=-1)]: Done   4 out of  19 | e

2025-11-26 22:38:37 - INFO - Saved Gaussian Copula model to GaussianCopulaTest/gaussiancopula42_noBR/GaussianCopula_model.pkl
2025-11-26 22:38:37 - INFO - Model fit complete.
2025-11-26 22:53:06 - INFO - Generated 121 synthetic samples with Gaussian Copula
2025-11-26 22:53:08 - INFO - Sampled 121 rows.
2025-11-26 22:53:10 - INFO - Anonymizing IDs in synthetic data
2025-11-26 22:53:10 - INFO - Applying rounding to synthetic data
2025-11-26 22:53:23 - INFO - Postprocessed synthetic data. Final shape: (121, 18815)
2025-11-26 22:53:35 - INFO - Saved synthetic data to GaussianCopulaTest/gaussiancopula42_noBR/gaussiancopula_42_noBR.csv
2025-11-26 22:53:35 - INFO - Pipeline complete.

--- Resource Usage Summary ---
Execution time: 2236.007525 seconds
Process RAM after execution: 3459.61 MB
Process RAM used by function: 2907.09 MB
Average per-core CPU during execution: [91.5, 3.7, 95.1, 0.8, 95.2, 1.0, 95.3, 0.6, 95.2, 100.0, 95.4, 0.2, 94.8, 97.3, 95.7, 97.3, 94.9, 97.1, 94.9, 32.1, 95.0, 0.7

In [9]:
fullDF["PD vs Responder"].value_counts()

PD vs Responder
Progressor    56
Responder     47
SD/MR         18
Name: count, dtype: int64

In [10]:
synthetic_data["PD vs Responder"].value_counts()

PD vs Responder
Progressor    56
Responder     46
SD/MR         19
Name: count, dtype: int64